In [40]:
import ollama
import json
import re

In [4]:
base_model_name = 'llama3.2:3b'
llama_model = 'llama3.2:3b'
phi_model = 'phi3:mini'
qwen_model = 'qwen3:4b'

## Helper Functions

In [17]:
def get_response(prompt: str, model_name=base_model_name):
    """
    Returns the response generated with the given prompt.
    """
    
    result = ollama.generate(model=model_name, prompt=prompt)

    return result['response']

In [30]:
def get_prompt_template(instruction: str, text: str, prompt_format: str = 'alpaca'):

    alpaca_format = '### Instruction:\n{}\n\n### Input:\n{}'
    chatml_format = '<|system|>\n{}<|end|>\n<|user|>\n{}<|end|>\n<|assistant|>'

    instruction = instruction.strip()
    text = text.strip()
    
    if prompt_format == 'alpaca':
        return alpaca_format.format(instruction, text)

    if prompt_format == 'chatml':
        return chatml_format.format(instruction, text)
        

In [74]:
def remove_think_tag(text: str): 
    return re.sub(r'<think>.*?</think>', '', text, flags=re.DOTALL).strip()

# Core Principles of Effective Prompting
The following are the core principles:
1. Write clear and specific instructions
2. Role Assignment
3. Separate system and user prompts
4. Format and output control
5. Step-by-step reasoning
6. Iterative prompt development

### Role Assignment

In [36]:
instruction = """You are a professional resume writer with 10+ years of experience helping software engineers land top tech 
jobs. Your task is to rewrite the following resume summary to make it more impactful and concise. No explanations.
"""

text = """A highly motivated and hardworking software engineer with experience in full-stack development, cloud deployment, and 
team collaboration. Passionate about writing clean code and solving real-world problems through technology.
"""

prompt = get_prompt_template(instruction, text)

print(f'Prompt: \n{prompt}')
print('----')
print(f'Response: ')
print(get_response(prompt))

Prompt: 
### Instruction:
You are a professional resume writer with 10+ years of experience helping software engineers land top tech 
jobs. Your task is to rewrite the following resume summary to make it more impactful and concise. No explanations.

### Input:
A highly motivated and hardworking software engineer with experience in full-stack development, cloud deployment, and 
team collaboration. Passionate about writing clean code and solving real-world problems through technology.
----
Response: 
Results-driven software engineer delivering scalable, high-quality solutions that drive business impact through collaborative team leadership and technical expertise.


In [37]:
# Phi-3 Mini model.
prompt = get_prompt_template(instruction, text, prompt_format='chatml')

print(f'Prompt: \n{prompt}')
print('----')
print(f'Response: ')
print(get_response(prompt, model_name=phi_model))

Prompt: 
<|system|>
You are a professional resume writer with 10+ years of experience helping software engineers land top tech 
jobs. Your task is to rewrite the following resume summary to make it more impactful and concise. No explanations.<|end|>
<|user|>
A highly motivated and hardworking software engineer with experience in full-stack development, cloud deployment, and 
team collaboration. Passionate about writing clean code and solving real-world problems through technology.<|end|>
<|assistant|>
----
Response: 
Dynamic Software Engineer proficient in full-stack development, cloud services integration, and fostering team productivity with a dedication to crafting maintainable code for practical solutions; eagerly seeking innovative roles where my commitment can drive success within the tech industry.


In [41]:
# Qwen model.print(f'Prompt: \n{prompt}')
prompt = get_prompt_template(instruction, text, prompt_format='alpaca')

print(f'Prompt: \n{prompt}')
print('----')
print(f'Response: ')
print(re.sub(r'<think>.*?</think>', '', get_response(prompt, model_name=qwen_model), flags=re.DOTALL).strip())

Prompt: 
### Instruction:
You are a professional resume writer with 10+ years of experience helping software engineers land top tech 
jobs. Your task is to rewrite the following resume summary to make it more impactful and concise. No explanations.

### Input:
A highly motivated and hardworking software engineer with experience in full-stack development, cloud deployment, and 
team collaboration. Passionate about writing clean code and solving real-world problems through technology.
----
Response: 
Driven software engineer with expertise in full-stack development, cloud deployment, and team collaboration, passionate about crafting clean, scalable code and driving impactful solutions through technology.


### Format and Output Control

In [45]:
instruction = """You are a smart and reliable assistant. Follow the instructions precisely. Always return a valid JSON.
If no books are found, return an empty JSON object {}.
If multiple books are found, return a list of JSON objects.
Each book should include only the following keys: id, book_name, and author.
Extract all books mentioned in the text below.
"""

text = """The book The Alchemist was written by Paulo Coelho. Jane Austen is the author of the book Emma. Norwegian Wood 
was penned by Haruki Murakami
"""

prompt = get_prompt_template(instruction, text)

print(f'Prompt: \n{prompt}')
print('----')
print(f'Response: ')
print(get_response(prompt))

Prompt: 
### Instruction:
You are a smart and reliable assistant. Follow the instructions precisely. Always return a valid JSON.
If no books are found, return an empty JSON object {}.
If multiple books are found, return a list of JSON objects.
Each book should include only the following keys: id, book_name, and author.
Extract all books mentioned in the text below.

### Input:
The book The Alchemist was written by Paulo Coelho. Jane Austen is the author of the book Emma. Norwegian Wood 
was penned by Haruki Murakami
----
Response: 
{"books": [{"id": "1", "book_name": "The Alchemist", "author": "Paulo Coelho"}, {"id": "2", "book_name": "Emma", "author": "Jane Austen"}, {"id": "3", "book_name": "Norwegian Wood", "author": "Haruki Murakami"}]}


### Step-by-Step Reasoning

In [48]:
instruction = """You are a helpful and intelligent assistant. Think step by step to solve the following problem. Show 
each step concisely and clearly and explain your reasoning before giving the final answer.
"""

text = """If a train travels 60 miles per hour for 2.5 hours, then stops for 30 minutes, and then continues at 40 miles per hour 
for another 1.5 hours, what is the total distance traveled and the total time (including the stop)?
"""

prompt = get_prompt_template(instruction, text)

print(get_response(prompt))

To solve this problem, we'll break it down into steps.

### Step 1: Calculate the distance traveled during the first part of the journey
Distance = Speed × Time
= 60 miles/hour × 2.5 hours
= 150 miles

### Step 2: Add the stop time to the total time
Total Time so far = 2.5 hours + 0.5 hours (30 minutes)
= 3 hours

### Step 3: Calculate the distance traveled during the second part of the journey
Distance = Speed × Time
= 40 miles/hour × 1.5 hours
= 60 miles

### Step 4: Add the distances traveled in both parts to find the total distance
Total Distance = 150 miles + 60 miles
= 210 miles

### Step 5: Add the remaining stop time and the last journey time to the total time
Remaining Stop Time = 0.5 hours (30 minutes)
Total Time so far = 3 hours
Total Time including stop = 3 hours + 0.5 hours
= 3.5 hours

### Final Answer:
The train traveled a total distance of **210 miles** and took a total time of **3.5 hours**, including the stop.


In [50]:
print(f'Prompt: \n{prompt}')
print('----')
print(f'Response: ')
print(get_response(prompt, model_name=qwen_model))

Prompt: 
### Instruction:
You are a helpful and intelligent assistant. Think step by step to solve the following problem. Show 
each step concisely and clearly and explain your reasoning before giving the final answer.

### Input:
If a train travels 60 miles per hour for 2.5 hours, then stops for 30 minutes, and then continues at 40 miles per hour 
for another 1.5 hours, what is the total distance traveled and the total time (including the stop)?
----
Response: 
<think>
Okay, let's see. I need to find the total distance traveled by the train and the total time including the stop. Let me break this down step by step.

First, the train travels at 60 miles per hour for 2.5 hours. To find the distance for that part, I should use the formula distance equals speed multiplied by time. So, 60 mph times 2.5 hours. Let me calculate that. 60 times 2 is 120, and 60 times 0.5 is 30, so adding those together gives 150 miles. So the first leg of the journey is 150 miles.

Then the train stops for 30 

# Types of Prompting Techniques
The following are the prompting techniques:
1. Zero-shot prompting
2. Few-shot prompting
3. Chain-of-thought prompting
4. Contextual Prompting

### Few-shot prompting

In [54]:
text = """
Fire : Hot :: Ice : ?
Answer: 
"""

instruction = f"""
You are a helpful assistant. Solve the analogy and no explanations but rewrite the line along with the missing value.

Example 1:
Man : King :: Woman : ?
Answer: Queen

Example 2:
Bird : Fly :: Fish : ?
Answer: Swim
"""

prompt = get_prompt_template(instruction, text)
print(get_response(prompt))

Fire : Hot :: Ice : Cold.


### Chain-of-Thought Prompting

In [63]:
text = """If there are 3 apples and you eat 1, how many are left?"""

instruction = f"""You are a helpful assistant. Think through the problem step-by-step using natural language chains 
such as "First..., then..., so...". Show your reasoning clearly, concisely, and return the final answer.
"""

prompt = get_prompt_template(instruction, text)
print(get_response(prompt))

To find out how many apples are left, let's break it down step by step:

First, we start with a certain number of apples: 3.

Then, we subtract the number of apples that were eaten: 1. This means we need to take away 1 apple from the original amount of 3 apples.

So, if we do the math: 3 (original apples) - 1 (apple eaten) = 2

Therefore, there are 2 apples left after you ate 1.


### Contextual Prompting

In [68]:
text = """"Okay team, let’s focus on the Q2 goals. First, we need to ship features faster — they don’t have to be perfect, but they
must deliver value. Also, we’ll split the $150K budget between frontend revamp and backend stability. Finally, QA needs 
more automation — we’re too dependent on manual tests."""

instruction = f"""You are an AI assistant helping summarize meeting transcripts. Given the transcript below, extract 
the top 3 key points discussed in the meeting concisely.
"""

context = """### Context:
The meeting was about Q2 project planning, including budget allocation, feature priorities, and inter-team 
coordination. The VP of Product emphasized speed over perfection. The Engineering team highlighted the need for better QA processes.
"""

prompt = get_prompt_template(instruction, text)
prompt = context + prompt

print(f'Prompt: \n{prompt}\n')
print('---')
print('Response:\n')
print(get_response(prompt))

Prompt: 
### Context:
The meeting was about Q2 project planning, including budget allocation, feature priorities, and inter-team 
coordination. The VP of Product emphasized speed over perfection. The Engineering team highlighted the need for better QA processes.
### Instruction:
You are an AI assistant helping summarize meeting transcripts. Given the transcript below, extract 
the top 3 key points discussed in the meeting concisely.

### Input:
"Okay team, let’s focus on the Q2 goals. First, we need to ship features faster — they don’t have to be perfect, but they
must deliver value. Also, we’ll split the $150K budget between frontend revamp and backend stability. Finally, QA needs 
more automation — we’re too dependent on manual tests.

---
Response:

Here are the top 3 key points discussed in the meeting:

1. **Ship features faster**: Prioritize speed over perfection to deliver value.
2. **Budget allocation**: Split $150K between frontend revamp and backend stability.
3. **Implement 

# Use cases of prompt engineering
The following are the prompting techniques:
1. Summarization
2. Inference
3. Expansion
4. Transformation
5. Translation
6. Question Answer Extraction
7. Data Formatting

### Summarization

In [71]:
text = """
Machine learning (ML) is a field of study in artificial intelligence concerned with the development and study of statistical 
algorithms that can learn from data and generalise to unseen data, and thus perform tasks without explicit instructions. Within 
a subdiscipline in machine learning, advances in the field of deep learning have allowed neural networks, a class of 
statistical algorithms, to surpass many previous machine learning approaches in performance.

ML finds application in many fields, including natural language processing, computer vision, speech recognition, email filtering, 
agriculture, and medicine. The application of ML to business problems is known as predictive analytics.

Statistics and mathematical optimisation (mathematical programming) methods comprise the foundations of machine learning. Data 
mining is a related field of study, focusing on exploratory data analysis (EDA) via unsupervised learning.

From a theoretical viewpoint, probably approximately correct learning provides a framework for describing machine learning.
"""

instruction = f"""You are an AI assistant helping summarize the technical text."""

prompt = get_prompt_template(instruction, text)
print(get_response(prompt))

Here's a summary of the technical text:

Machine learning (ML) is an AI field that enables algorithms to learn from data and generalize to new situations, performing tasks without explicit instructions. Advances in deep learning have improved performance in various areas like natural language processing, computer vision, and more. ML has numerous applications across fields such as agriculture, medicine, and business, using techniques like predictive analytics. The foundation of ML is rooted in statistics and mathematical optimization methods, with data mining being a related field that focuses on exploratory data analysis.


In [75]:
print(remove_think_tag(get_response(prompt, model_name=qwen_model)))

**Summary:**  
Machine learning (ML) is a subfield of artificial intelligence focused on statistical algorithms that learn from data and generalize to new situations, operating without explicit instructions. Advances in deep learning have enabled neural networks to outperform traditional ML methods. ML is applied across domains such as natural language processing, computer vision, speech recognition, agriculture, medicine, and business (predictive analytics). Its foundations include statistics, mathematical optimization, and data mining, which emphasizes exploratory data analysis via unsupervised learning. Theoretical frameworks like Probably Approximately Correct (PAC) learning provide a basis for understanding ML's learning processes.


In [72]:
prompt = get_prompt_template(instruction, text, prompt_format='chatml')
print(get_response(prompt, model_name=phi_model))

Machine Learning (ML), as an integral component of artificial intelligence research and application, focuses on creating algorithms capable of deriving patterns from data to generalize beyond their initial training set into unseen scenarios without direct human guidance. Deep learning—a subset within ML that leverages complex neural networks—has shown remarkable progress in surpassing traditional methods across various domains such as natural language processing (NLP), computer vision, speech recognition, email spam filtering, agriculture, and medical diagnostics through predictive analytics aimed at solving business problems. The foundational bedrock of ML lies within statistical algorithms underpinned by statistics and mathematical programming techniques used for data mining via unsupervised learning approaches during exploratory data analysis (EDA). From a theoretical standpoint in the field, Probably Approximately Correct Learning offers an overarching framework to describe machine

### Inference

#### Main topic and Subtopic Identification

In [77]:
text = "GPT-4 can now process images, making it a powerful tool in AI-driven content moderation."

instruction = f"""You are a helpful and intelligent assistant. Follow instructions exactly. Return only a valid JSON object 
with the keys topic and subTopic. No explanations and No extra text or formatting.
"""

prompt = get_prompt_template(instruction, text)
print(get_response(prompt))

{"topic": "AI", "subTopic": "Computer Vision"}


#### Named Entity Recognition

In [81]:
text = """Apple Inc. unveiled the new iPhone 15 Pro Max during its annual event on September 12, 2023, in 
Cupertino, California. Tim Cook introduced the device, which costs $1,199, and emphasized its titanium frame and advanced AI 
features. Analysts expect a 10% increase in sales compared to last year."""

instruction = f"""You are a helpful assistant trained in Natural Language Processing. Extract all named entities from the given 
text and categorize them into the following types: PERSON, ORGANIZATION, LOCATION. No explanations or extra text.

Return the output as a JSON array of objects with keys: "text", "type". 

If there are no entities, return an empty array.

"""

prompt = get_prompt_template(instruction, text)
print(get_response(prompt))

[
    {"text": "Apple Inc.", "type": "ORGANIZATION"},
    {"text": "Tim Cook", "type": "PERSON"},
    {"text": "Cupertino", "type": "LOCATION"},
    {"text": "California", "type": "LOCATION"}
]


#### Sentiment Analysis

In [82]:
text = """I recently bought the Lumina Smart Speaker, and I must say, the sound quality is absolutely incredible! The voice 
assistant is quick to respond and understands my commands perfectly. However, the setup process was a bit confusing at first.
"""

instruction = f"""You are a sentiment analysis assistant. Analyze the sentiment of the given text and return one of the 
following labels: "positive", "negative", or "neutral".
"""

prompt = get_prompt_template(instruction, text)
print(get_response(prompt))

The sentiment of the given text is: "positive"

Although the text mentions that the setup process was a bit confusing, the overall tone and language used (e.g., "absolutely incredible", "quick to respond") are overwhelmingly positive, indicating that the speaker's strengths outweigh its weaknesses in this case.


### Expanding

In [86]:
sentiment = 'neutral'

text = """
I recently purchased a pair of wireless earbuds and overall, they’ve been a solid addition to my daily routine. I really appreciate
how compact and lightweight they are — they fit comfortably in my ears and are easy to carry around, even in a small pocket 
or bag. The sound quality is quite decent for casual listening — vocals are clear, and the bass, while not booming, is still 
present enough for most genres of music.

However, I’ve noticed that the battery life isn’t as long-lasting as I had hoped. I find myself needing to recharge them 
more often than expected, especially if I’m using them for long calls or extended music sessions throughout the day. It’s not 
a dealbreaker, but definitely something to be aware of for those who need all-day use.

Overall, I’m satisfied with the product — it offers convenience and acceptable performance at a reasonable price point, but 
there’s room for improvement in terms of battery longevity.
"""


instruction = f"""You are a customer service AI assistant.
Your task is to send an email reply to a valued customer.
Given the customer review delimited first expand the short feedback into a detailed interpretation
to better understand the customer experience. Then generate a
reply to thank the customer for their review.

If the sentiment is positive or neutral, thank them for
their review and acknowledge their input.
If the sentiment is negative, apologize and suggest that
they can reach out to customer service. 
Make sure to reference specific product aspects from the review.
Write in a concise and professional tone.
Sign the email as `AI customer agent`.

Review sentiment: {sentiment}
"""

prompt = get_prompt_template(instruction, text)

print(remove_think_tag(get_response(prompt, model_name=qwen_model)))

**Subject:** Thank You for Your Feedback  

Dear Valued Customer,  

Thank you for taking the time to share your experience with our wireless earbuds. We appreciate your detailed review and are glad to hear that the product has met your expectations in terms of design and sound quality.  

We particularly appreciate your comments on the compact and lightweight design, which allows for easy portability, and the clear vocals and balanced bass that suit a variety of music genres. These are key aspects of our product development, and it’s reassuring to know they align with your needs.  

We also recognize your note about battery life and understand that this can be a consideration for extended use. While we strive to optimize performance, we welcome your feedback to help us improve further. If you have additional questions or need assistance, please don’t hesitate to reach out to our customer service team.  

Thank you again for your thoughtful review. We’re committed to ensuring our produ

### Transformation

In [90]:
text = "안녕하세요. 오늘 날씨가 정말 좋네요"

instruction = "You are an expert language translator. Translate the following text from Korean to English. No explanations"

prompt = get_prompt_template(instruction, text)

print(remove_think_tag(get_response(prompt, model_name=qwen_model)))

Hello. Today's weather is really good.


In [91]:
print(get_response(prompt))

Hello, today's weather is really nice.


### Question Answering

In [93]:
text = "What are the three main types of machine learning?"

instruction = """You are a helpful assistant. Use the given context to answer the question. If the answer is not in the context, 
respond with "Not found in the context.

### Context
Machine learning is a subfield of artificial intelligence that focuses on the development of algorithms that can learn 
from data and make predictions or decisions without being explicitly programmed. Supervised learning, unsupervised learning, 
and reinforcement learning are the three main types of machine learning.
"""

prompt = get_prompt_template(instruction, text)

print(prompt)
print(get_response(prompt))

### Instruction:
You are a helpful assistant. Use the given context to answer the question. If the answer is not in the context, 
respond with "Not found in the context.

### Context
Machine learning is a subfield of artificial intelligence that focuses on the development of algorithms that can learn 
from data and make predictions or decisions without being explicitly programmed. Supervised learning, unsupervised learning, 
and reinforcement learning are the three main types of machine learning.

### Input:
What are the three main types of machine learning?
According to the context, the three main types of machine learning are:

1. Supervised learning
2. Unsupervised learning
3. Reinforcement learning


### Data Formatting


In [100]:
text = """The book The Alchemist was written by Paulo Coelho. Jane Austen is the author of the book Emma. Norwegian 
Wood was penned by Haruki Murakami"""

instruction = """You are a precise and intelligent assistant. Extract all books mentioned in the text. Return only valid JSON 
with each book as an object containing the keys: id, book_name, and author.

If no books are found, return an empty JSON object: {}. If multiple books are present, return a list of objects. Do not include 
any extra text. 
"""

prompt = get_prompt_template(instruction, text)
print(get_response(prompt))

{"books": [{"id": "1", "book_name": "The Alchemist", "author": "Paulo Coelho"}, {"id": "2", "book_name": "Emma", "author": "Jane Austen"}, {"id": "3", "book_name": "Norwegian Wood", "author": "Haruki Murakami"}]}


## More examples

In [23]:
user_input = """Prompt Engineering for Plumbers is not a book written by John Doe"""

prompt = f"""
Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
You are a helpful and intelligent assistant. Always follow the instruction strictly. Only return 
in the valid JSON format. If no data is present then return empty JSON object ony.

Return all the books mentioned in the string delimited by triple backticks with keys id, book_name, and author as list of objects and nothing else.

### Input:
```{user_input}```[/INST]

### Response:
"""

response = get_response(prompt)

print(response)

{"books": []}


### Numerical Entity Extraction


In [24]:
user_input = 'The RTX 4090 GPU has 16384 CUDA cores, 24 GB VRAM, and a power draw of 450 watts. It was launched at $1599 in 2022.'

In [25]:
prompt = f"""
Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
You are a helpful and intelligent assistant. Always follow the instruction strictly. Only return the numbers as a list. What numbers are 
present within the string delimited by triple backticks?

### Input:
```{user_input}```

### Response:
"""

print(get_response(prompt))

[16384, 24, 450, 1599]


### Text Style Transformation

In [50]:
user_input = "Python was created by Guido van Rossum in 1991. As of 2023, it remains one of the most used programming \
languages in the world. In the last Python Developer Survey, over 80% of respondents said they use it daily. The core team \
includes developers from Google, Microsoft, and independent contributors. Guido, now working at Microsoft, still occasionally \
comments on PEP discussions."

prompt = f"""
Below is an instruction that describes a task. Write a response that appropriately completes the request.

### Instruction:
You are a helpful and intelligent assistant. Always follow the instruction strictly. Transform the following text into points
as a list. Return a valid JSON object with key "transformation" and value as a list of points from the text.

No explanations and backticks

### Input:
```{user_input}``` 
"""

response = get_response(prompt)

try:
    print(json.dumps(json.loads(response), indent=4))
except:
    print(response)


{
    "transformation": [
        "Python was created by Guido van Rossum in 1991.",
        "As of 2023, it remains one of the most used programming languages in the world.",
        "In the last Python Developer Survey, over 80% of respondents said they use it daily.",
        "The core team includes developers from Google, Microsoft, and independent contributors.",
        "Guido, now working at Microsoft, still occasionally comments on PEP discussions."
    ]
}


In [56]:
len(user_input.split())

144

In [57]:
len(response.split())

75

### Few-shot prompting

In [63]:
user_input = """
Fire : Hot :: Ice : ?
Answer:
"""

prompt = f"""
### Instruction:
You are a helpful assistant. Solve the analogy and no explanations but rewrite the line along with the missing value.

Example 1:
Man : King :: Woman : ?
Answer: Queen

Example 2:
Bird : Fly :: Fish : ?
Answer: Swim

### Input:
```{user_input}``` 
"""

response = get_response(prompt)

try:
    print(json.dumps(json.loads(response), indent=4))
except:
    print(response)


Fire : Hot :: Ice : Cold


In [67]:
review = """
The RTX 4060 Ti is a decent card if you're targeting 1080p or light 1440p gaming. It’s very power-efficient and stays cool even under load, which is great for smaller builds. DLSS 3 also helps boost performance in supported games, making demanding titles much more playable.

However, the 8GB VRAM is starting to show its age, especially in newer games like The Last of Us Part I and Resident Evil 4 Remake. At this price point, I expected more longevity. It performs well, but it feels like NVIDIA could've offered better specs without increasing the cost. It’s not a bad card, but it’s hard to recommend over the competition unless power draw and DLSS are your top priorities.
"""

prompt = f"""
### Instruction:
You are a helpful assistant. Your task is find the product name and comapny name from the given customer review delimited by triple
backticks. Return the valid JSON with keys product_name, and company_name only. No explanations or unnecessary text or backticks.

### Input:
```{review}``` 
"""

response = get_response(prompt)

try:
    print(json.dumps(json.loads(response), indent=4))
except:
    print(response)


{
    "product_name": "RTX 4060 Ti",
    "company_name": "NVIDIA"
}


In [73]:
product_feedback = """
Customer Feedback for Dell XPS 15 9530 (2023, i7-13700H, RTX 4060, 32GB RAM)

I’ve been using the Dell XPS 15 9530 for about 3 months now, and overall, I’d say it’s a strong contender for a premium productivity and light gaming laptop, though not without its flaws. I primarily use it for software development, some 3D modeling (Blender), and casual AAA gaming. I’ll break down my thoughts into various aspects:

Performance (9/10):
With the Intel i7-13700H and the RTX 4060 GPU, this laptop absolutely flies through multi-threaded tasks like code compilation and moderate 3D rendering. Running Docker containers and VS Code with multiple extensions, along with a few Chrome tabs, doesn’t slow it down one bit. Blender renders are fast, although the GPU tends to throttle slightly when plugged in without “Ultra Performance” mode in Dell’s Power Manager. Gaming-wise, it handles Cyberpunk 2077 on medium-high settings at 1080p with DLSS enabled, maintaining 60+ FPS.

Build Quality & Design (8.5/10):
The design is still one of the best in the Windows ecosystem—clean, minimal, and premium. The CNC aluminum chassis and carbon fiber palm rest feel excellent. However, it is a fingerprint magnet, especially around the keyboard area. It’s also a bit on the heavier side at ~4.2 lbs, so not the most portable option for travel.

Display (10/10):
The 3.5K OLED touch display is stunning—vivid colors, deep blacks, and excellent contrast. It’s factory-calibrated and ideal for color-accurate work. One minor gripe: it’s glossy and very reflective, so working outdoors or in direct sunlight is a bit challenging.

Keyboard & Trackpad (8/10):
The keyboard has decent travel and feels responsive, but I still prefer the typing feel of my ThinkPad T14 for longer sessions. The trackpad is massive and accurate but can sometimes misregister palm input despite palm rejection being enabled.

Thermals & Fan Noise (7/10):
Thermal performance is acceptable but not great. Under load, it can get toasty—CPU temps often hit 90°C+, and the fans spin up noticeably. It's not super loud, but definitely audible in a quiet room. When idle or doing light tasks, it remains silent and cool.

Battery Life (7.5/10):
With the 86Wh battery, I average around 6 to 7 hours on light workloads—web browsing, Slack, VS Code, and YouTube at 50% brightness. Not amazing, but decent for a power-hungry OLED and H-series CPU. Gaming and heavy use drain it fast—down to about 90 minutes of runtime.

Ports & Connectivity (8/10):
Three USB-C/Thunderbolt ports, one full-sized SD card reader, and a headphone jack. No USB-A, but Dell includes a dongle. Wi-Fi 6E and Bluetooth 5.3 are solid—no drops or issues. I do miss having an Ethernet port, though.

Software & Bloat (6/10):
Dell’s pre-installed software like Dell Power Manager and SupportAssist is functional but intrusive. I had to disable several background tasks to prevent unnecessary CPU spikes. Also, McAfee came pre-installed—first thing I uninstalled.

Pricing & Value (7.5/10):
I paid around $2,399 for this configuration. It’s expensive, no doubt. Comparable devices like the MacBook Pro 16 offer better battery and thermal performance, but you lose the gaming capability. For Windows power users who need a balance of portability, power, and style, this is a reasonable investment—but definitely not a budget-friendly pick.

Final Verdict:
The Dell XPS 15 9530 is a powerhouse in a sleek chassis with a jaw-dropping display. It's nearly perfect for professionals who dabble in creative workloads and occasional gaming. However, thermal limitations and bloatware slightly detract from the experience. If Dell improved the thermal design and reduced pre-installed software, this would be a 9.5/10 device.

Rating: 8.5/10
"""

prompt = f"""
### Instruction:
You are a helpful assistant aiding the product marketing team. Read the customer feedback and extract:
1. Key strengths of the product
2. Pain points or issues mentioned
3. Specific suggestions or implied areas of improvement

Format the output clearly using bullet points. Be concise and avoid any explanations.

### Input:
```{product_feedback}```
"""

print(get_response(prompt))

**Key Strengths of the Product:**

• Excellent display with vivid colors, deep blacks, and excellent contrast
• Fast performance with Intel i7-13700H and RTX 4060 GPU
• Premium build quality and design
• Good keyboard feel and responsive trackpad

**Pain Points or Issues Mentioned:**

• Thermal limitations and hot temperatures under heavy load
• Fingerprint magnet and heavy weight
• Minor issues with battery life (not amazing, but decent)
• Noisy fans during intense use
• No Ethernet port
• Bloatware from pre-installed software

**Specific Suggestions or Implied Areas of Improvement:**

• Improved thermal design to reduce hot temperatures under load
• Reduced bloatware and intrusive pre-installed software
• Enhanced palm rejection on the trackpad
• More portable design options (e.g., lighter weight)
• Alternative ports (e.g., Ethernet port, USB-A)


In [74]:
prompt = f"""
### Instruction:
Extract all **technical observations and metrics** mentioned in the user feedback. These include performance stats, temperatures, battery timings, frame rates, hardware details, etc.

List them in bullet points. No interpretations or summaries.

### Input:
```{product_feedback}```
"""

print(get_response(prompt))

Here are the technical observations and metrics mentioned in the user feedback:

• Performance stats:
  • Multi-threaded tasks: "absolutely flies"
  • Code compilation: fast
  • Moderate 3D rendering: fast (with GPU throttle)
  • Gaming at 1080p with DLSS enabled: 60+ FPS
• Temperatures:
  • CPU temps: often hit 90°C+
• Hardware details:
  • Intel i7-13700H CPU
  • RTX 4060 GPU
  • 32GB RAM
• Battery timings:
  • Battery capacity: 86Wh
  • Average runtime on light workloads: 6 to 7 hours
  • Runtime on heavy use: about 90 minutes
• Display metrics:
  • Resolution: 3.5K OLED touch display
• Frame rates:
  • Gaming at 1080p with DLSS enabled: 60+ FPS
